# 📊 Auditoría Masiva de Facturación 2026

Este notebook tiene dos objetivos principales:
1. **Análisis de Frecuencia:** Analizar el historial de 2026 desde el CSV para identificar la frecuencia de facturación de cada proveedor.
2. **Simulación de Renombrado:** Tomar una muestra de 1,500 PDFs de la carpeta de 2026, pasarlos por la IA extractora del robot, y comparar el nombre propuesto vs el nombre real del archivo.

In [ ]:
import os
import random
import fitz
import pandas as pd
import sys

# Asegurarnos de poder importar procesador_core
sys.path.append(r'c:\Users\gcastro\ROBOTS\asistentecontable\Proyecto_facturas_v2')
from procesador_core import extraer_datos_factura

import warnings
warnings.filterwarnings("ignore")

## 1. Análisis de Frecuencias Históricas (Desde CSV)

In [ ]:
ruta_csv = r"C:\Users\gcastro\Downloads\2026 FC(TELCOM).csv"

try:
    df_hist = pd.read_csv(ruta_csv, sep=';', encoding='latin1')
    
    # Limpiar columnas innecesarias (las Unnamed que se crean por exceso de punto y coma)
    df_hist = df_hist.loc[:, ~df_hist.columns.str.contains('^Unnamed')]
    
    print(f"✅ CSV cargado correctamente con {len(df_hist)} registros.")
    
    # Limpiar nombres de proveedores para unificarlos
    if 'PROVEEDOR' in df_hist.columns:
        df_hist['Proveedor Limpio'] = df_hist['PROVEEDOR'].astype(str).str.strip().str.upper()
        # Agrupar por proveedor
        frecuencias = df_hist.groupby('Proveedor Limpio').size().reset_index(name='Total Facturas 2026')
        frecuencias = frecuencias.sort_values(by='Total Facturas 2026', ascending=False)
        
        display(frecuencias.head(15))
    else:
        print("No se encontró la columna PROVEEDOR en el CSV.")
        
except Exception as e:
    print(f"❌ Error al leer el CSV: {e}")

## 2. Auditoría Masiva de PDFs (Simulación)

In [ ]:
CARPETA_2026 = r"\\10.10.10.210\\AyF_Trabajoadistancia\\Facturas a pagar\\2026"
MUESTRA_MAXIMA = 1500

def obtener_texto_pdf(ruta_pdf):
    """Extrae texto de las primeras dos páginas"""
    try:
        doc = fitz.open(ruta_pdf)
        texto = ""
        for page_num in range(min(2, doc.page_count)):
            texto += doc.load_page(page_num).get_text()
        doc.close()
        return texto
    except:
        return ""

print("🔍 Escaneando carpeta para encontrar PDFs...")
todos_los_pdfs = []
for root, dirs, files in os.walk(CARPETA_2026):
    for file in files:
        if file.lower().endswith('.pdf'):
            todos_los_pdfs.append(os.path.join(root, file))

print(f"Encontrados {len(todos_los_pdfs)} PDFs en total.")

# Tomar muestra aleatoria si hay más del máximo
if len(todos_los_pdfs) > MUESTRA_MAXIMA:
    muestra_pdfs = random.sample(todos_los_pdfs, MUESTRA_MAXIMA)
else:
    muestra_pdfs = todos_los_pdfs

print(f"Procesando {len(muestra_pdfs)} PDFs... Esto tomará un par de minutos.")

In [ ]:
resultados = []

for i, ruta in enumerate(muestra_pdfs):
    if i > 0 and i % 100 == 0:
        print(f"... procesados {i}/{len(muestra_pdfs)} archivos")
        
    nombre_original = os.path.basename(ruta)
    texto = obtener_texto_pdf(ruta)
    
    if not texto.strip():
        resultados.append({
            "Nombre Original": nombre_original,
            "Proveedor Propuesto": "ERROR_LECTURA",
            "Nombre Propuesto": "NO_SE_PUDO_LEER_TEXTO"
        })
        continue
        
    # Usar el cerebro del robot
    datos_extraidos = extraer_datos_factura(texto, "telcom", nombre_original)
    
    proveedor = datos_extraidos.get("nombre", "DESCONOCIDO")
    tipo_num = datos_extraidos.get("tipo_y_numero", "S/N")
    
    # Construir el nombre que propondría el robot (sin prefijo de mes para comparar fácil)
    # En el robot real, el sector y el mes los pone el usuario. 
    # Aquí solo comparamos si el Proveedor y Número están bien.
    nombre_propuesto = f"{proveedor} {tipo_num}.pdf".upper()
    
    resultados.append({
        "Nombre Original": nombre_original,
        "Proveedor Propuesto": proveedor,
        "Nombre Propuesto (Base)": nombre_propuesto,
        "Coincide Proveedor en Nombre?": proveedor.upper() in nombre_original.upper() if proveedor != "DESCONOCIDO" else False
    })

print("✅ Procesamiento finalizado.")

In [ ]:
df_res = pd.DataFrame(resultados)

# Mostrar resumen de precisión
tasa_acierto = (df_res['Coincide Proveedor en Nombre?'].sum() / len(df_res)) * 100
print(f"\n🎯 Tasa de acierto (El proveedor propuesto por IA aparece en el nombre original): {tasa_acierto:.1f}%")

# Mostrar 20 archivos al azar donde la IA difiere del nombre original
display(df_res[df_res['Coincide Proveedor en Nombre?'] == False].sample(min(20, len(df_res[df_res['Coincide Proveedor en Nombre?'] == False]))))